# YOLOv8-cls Download and Keras 3.0 Conversion

This notebook downloads the YOLOv8 classification model, exports it to ONNX, sanitizes tensor names for Keras 3.0 compliance, and converts it to Keras.

In [1]:
import os
import onnx
from ultralytics import YOLO
from onnx2keras.onnx2keras import onnx_to_keras

In [2]:
def sanitize_onnx_model(model):
    """
    Sanitizes ONNX model tensor names by replacing '/' and ':' with '_' to comply with Keras 3.0.
    """
    def rename(name):
        return name.replace('/', '_').replace(':', '_')
        
    for inp in model.graph.input:
        inp.name = rename(inp.name)
    for out in model.graph.output:
        out.name = rename(out.name)
    for init in model.graph.initializer:
        init.name = rename(init.name)
    for node in model.graph.node:
        if node.name:
            node.name = rename(node.name)
        inputs = [rename(i) for i in node.input]
        del node.input[:]
        node.input.extend(inputs)
        
        outputs = [rename(o) for o in node.output]
        del node.output[:]
        node.output.extend(outputs)
    for vi in model.graph.value_info:
        vi.name = rename(vi.name)
    return model

In [3]:
# Step 1: Download and load YOLOv8n-cls model
model = YOLO('yolov8n-cls.pt')

In [4]:
# Step 2: Export model to ONNX
onnx_path = model.export(format='onnx', imgsz=224)
print(f'ONNX model exported to: {onnx_path}')

Ultralytics 8.4.87  Python-3.13.1 torch-2.12.1+cpu CPU (Intel Core i5-10300H 2.50GHz)


YOLOv8n-cls summary (fused): 30 layers, 2,715,880 parameters, 0 gradients, 4.3 GFLOPs



PyTorch: starting from 'yolov8n-cls.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 1000) (5.3 MB)



ONNX: starting export with onnx 1.22.0 opset 20...


ONNX: slimming with onnxslim 0.1.94...


ONNX: export success  0.6s, saved as 'yolov8n-cls.onnx' (10.4 MB)



Export complete (0.8s)
Results saved to E:\phenikaa\quant_anything\yolov8n-cls.onnx
Predict:         yolo predict task=classify model=yolov8n-cls.onnx imgsz=224 
Validate:        yolo val task=classify model=yolov8n-cls.onnx imgsz=224 data=../datasets/imagenet  
Visualize:       https://netron.app


ONNX model exported to: yolov8n-cls.onnx


In [5]:
# Step 3: Load ONNX model and convert to Keras
onnx_model = onnx.load(onnx_path)
sanitized_onnx = sanitize_onnx_model(onnx_model)
keras_model = onnx_to_keras(
    sanitized_onnx,
    input_names=['images'],
    input_shapes=[[3, 224, 224]],
    verbose=True,
    change_ordering=False
)
keras_model.summary()

INFO:onnx2keras:Converter is called.


DEBUG:onnx2keras:List input shapes:


DEBUG:onnx2keras:[[3, 224, 224]]


DEBUG:onnx2keras:List inputs:


DEBUG:onnx2keras:Input 0 -> images.


...

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, 3, 224,    │          0 │ -                 │
│                     │ 224)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.0_conv_Conv… │ (None, 3, 226,    │          0 │ images[0][0]      │
│ (ZeroPadding2D)     │ 226)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.0_conv_Conv… │ (None, 16, 112,   │        448 │ _model.0_conv_Co… │
│ (Conv2D)            │ 112)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.0_act_Sigmo… │ (None, 16, 112,   │          0 │ _model.0_conv_Co… │
│ (Activation)        │ 112)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.0_act_Mul_o… │ (None, 16, 112,   │          0 │ _model.0_conv_Co… │
│ (Multiply)          │ 112)              │            │ _model.0_act_Sig… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.1_conv_Conv… │ (None, 16, 114,   │          0 │ _model.0_act_Mul… │
│ (ZeroPadding2D)     │ 114)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.1_conv_Conv… │ (None, 32, 56,    │      4,640 │ _model.1_conv_Co… │
│ (Conv2D)            │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.1_act_Sigmo… │ (None, 32, 56,    │          0 │ _model.1_conv_Co… │
│ (Activation)        │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.1_act_Mul_o… │ (None, 32, 56,    │          0 │ _model.1_conv_Co… │
│ (Multiply)          │ 56)               │            │ _model.1_act_Sig… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_cv1_conv_… │ (None, 32, 56,    │      1,056 │ _model.1_act_Mul… │
│ (Conv2D)            │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_cv1_act_S… │ (None, 32, 56,    │          0 │ _model.2_cv1_con… │
│ (Activation)        │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_cv1_act_M… │ (None, 32, 56,    │          0 │ _model.2_cv1_con… │
│ (Multiply)          │ 56)               │            │ _model.2_cv1_act… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_Split_out… │ (None, 16, 56,    │          0 │ _model.2_cv1_act… │
│ (Lambda)            │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_m.0_cv1_c… │ (None, 16, 58,    │          0 │ _model.2_Split_o… │
│ (ZeroPadding2D)     │ 58)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_m.0_cv1_c… │ (None, 16, 56,    │      2,320 │ _model.2_m.0_cv1… │
│ (Conv2D)            │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_m.0_cv1_a… │ (None, 16, 56,    │          0 │ _model.2_m.0_cv1… │
│ (Activation)        │ 56)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ _model.2_m.0_cv1_a… │ (None, 16, 56,    │          0 │ _model.2_m.0_cv1

 Total params: 2,715,880 (10.36 MB)

 Trainable params: 2,715,880 (10.36 MB)

 Non-trainable params: 0 (0.00 B)